In [0]:
import random
from datetime import datetime

# 1. Configuration - Use the same Bronze folder as your historical logs
# This keeps all 'unstructured' raw data in one landing zone
output_folder = "abfss://battery-data@batteryhealthdatalake.dfs.core.windows.net/bronze/realtime_bms_logs/"

# Real-time fragments (Technical, no 'normal/danger' labels)
realtime_fragments = [
    "THERMAL_RUNAWAY_PREDICT: Gradient {t_grad}C/s at sensor_03",
    "VOLT_SKEW: Cell_09 mismatch {diff}V vs Pack_Avg",
    "RES_IMPEDANCE_ERR: High resistance {res}mOhm on Busbar_2",
    "SIGNAL_NOISE: EMI interference detected on BMS_CAN_BUS",
    "OCV_RECOVERY_DELAY: Voltage recovery slower than {sag}V/sec"
]

# 2. Generate a "Burst" of 10 log entries for RIGHT NOW
log_time_str = datetime.now().strftime('%Y%m%d_%H%M%S')
filename = f"realtime_bms_log_{log_time_str}.txt"
full_path = output_folder + filename

log_content = f"--- BMS REALTIME LOG BURST: {log_time_str} ---\n"

for _ in range(10):
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    log_line = random.choice(realtime_fragments).format(
        diff=round(random.uniform(0.3, 0.6), 3),
        res=round(random.uniform(10.0, 20.0), 1),
        t_grad=round(random.uniform(1.2, 3.5), 2),
        sag=round(random.uniform(0.5, 1.5), 2)
    )
    log_content += f"[{timestamp}] {log_line}\n"

# 3. Write to Azure (Cost: $0.00 in credits, only minimal storage fee)
dbutils.fs.put(full_path, log_content, overwrite=True)

print(f"Successfully sent realtime log burst to: {full_path}")